# Remaining runs — statistical aggregation, convergence, paired inference

Status at start:
- MovieLens `ml-1m/ratings.dat` present (`paper-ideas/ActionShap/code/data/ml-1m/ratings.dat`).
- Seed results present: `results/raw/movielens_actionshap_seed{42-46}.json`.
- Full-catalogue results present: `results/raw/fullcatalog_actionshap_seed{42-46}.json`.
- Convergence result present: `results/raw/convergence_seed42.json`.
- Manifest (`paper/manifests/asset_manifest.json`): `PASS`, seeds `[42,43,44,45,46]`, errors `[]`, warnings `[]`.
- Notebook `ActionShap_All.ipynb`: cell 5 valid (seed loop); cells 23/44/45 filled.

This notebook completes what the manifest notes explicitly: paired user-level inference (`paper/tables/paired_tests.csv`) and full convergence analysis must be finalized before any claim is treated as confirmed.

## 1. Aggregate per-user seed results into a single evaluation table

In [ ]:
import json, pathlib, pandas as pd, numpy as np
from collections import Counter

ROOT = pathlib.Path('paper-ideas/ActionShap/code/notebook_remain.ipynb').resolve().parent.parent.parent.parent  # repo root
RAW = ROOT / 'paper-ideas' / 'ActionShap' / 'code' / 'results' / 'raw'
OUT = ROOT / 'paper-ideas' / 'ActionShap' / 'code' / 'results' / 'aggregated'
OUT.mkdir(exist_ok=True)

files = sorted(RAW.glob('movielens_actionshap_seed*.json'))
print('Found seed files:', [f.name for f in files])

def load_metrics(p):
    payload = json.loads(p.read_text())
    if isinstance(payload, list):
        payload = payload[0] if payload else {}
    metrics = payload.get('metrics', {})
    users = payload.get('users', [])
    dataset = payload.get('dataset', payload.get('config', {}))
    seed = int(p.stem.replace('movielens_actionshap_seed', '').replace('fullcatalog_actionshap_seed', ''))
    return {
        'seed': seed,
        'users_n': len(users) if isinstance(users, list) else users,
        'dataset_key': str(dataset.get('dataset', 'unknown')),
        'metrics_summary': metrics
    }

rows = [load_metrics(f) for f in files]
summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUT / 'seed_summary.csv', index=False)
print('Wrote aggregated seed summary:', (OUT / 'seed_summary.csv').relative_to(ROOT))
print('Rows:', len(summary_df))
display(summary_df)


## 2. Load per-user metrics across seeds for paired inference

In [ ]:
user_rows = []
for f in sorted(RAW.glob('movielens_actionshap_seed*.json')):
    payload = json.loads(f.read_text())
    if isinstance(payload, list):
        payload = payload[0] if payload else {}
    seed = int(f.stem.split('seed')[-1])
    metrics = payload.get('metrics', {})
    user_list = payload.get('users', []) if 'users' in payload else list(range(metrics.get('n_users', 500)))
    aia_mean = metrics.get('aia_mean')
    regret_mean = metrics.get('regret_mean')
    user_rows.append({
        'seed': seed,
        'file': f.name,
        'users_n': len(user_list) if isinstance(user_list, list) else user_list,
        'aia_mean': aia_mean,
        'regret_mean': regret_mean,
        'dataset': payload.get('dataset', payload.get('config', {})).get('dataset', 'unknown')
    })
paired_df = pd.DataFrame(user_rows)
paired_df.to_csv(OUT / 'paired_source.csv', index=False)
print('Paired source rows:', len(paired_df))
display(paired_df)


## 3. Verify convergence file and propose regeneration check

In [ ]:
conv_path = RAW / 'convergence_seed42.json'
if conv_path.exists():
    payload = json.loads(conv_path.read_text())
    print('Convergence file present:', payload.get('file') or payload.get('seed') or 'seed42')
    print('Top-level keys:', list(payload.keys()) if isinstance(payload, dict) else 'list')
else:
    print('Convergence file NOT present; fig06 remains pending.')

manifest_path = ROOT / 'paper-ideas' / 'ActionShap' / 'paper' / 'manifests' / 'asset_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print('Manifest status:', manifest.get('validation', {}).get('status'))
    print('Seeds listed:', manifest.get('seeds'))
else:
    print('No manifest at', manifest_path)


## 4. What remains explicitly pending (recorded for transparency)

In [ ]:
pending_items = []
pending_items.append('Paired user-level statistical inference table (paper/tables/paired_tests.csv) not yet produced from aggregated results.')
pending_items.append('Full convergence statistical report beyond seed42 not produced.')
pending_items.append('Paper claims remain descriptive, not final inferential; paired tests and full convergence must finalize.')
print('\n'.join(pending_items))
